In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [6]:
# Define the path to your F1 PDF
PDF_PATH = "C:/Users/Noel/Documents/Personal Projects/f1-rag-knowledge-assistant/data/F1_2024_BritishGP_RaceReport.pdf"

# Create the loader
loader = PyPDFLoader(PDF_PATH)

# Load the document pages
pages = loader.load()

# See what we got
print(f"Total pages loaded: {len(pages)}")
print(f"\nFirst page preview:\n{pages[0].page_content[:500]}")

Total pages loaded: 7

First page preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2
This docum


In [7]:
#Create the splitters
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

#Split the pages into chunks
chunks = splitter.split_documents(pages)

print(f"total chunks created :{len(chunks)}")
print(f"\nFirst chunk preview:\n{chunks[0].page_content}")

total chunks created :16

First chunk preview:
FIA FORMULA ONE WORLD CHAMPIONSHIP
 
 2024 FORMULA 1
QATAR AIRWAYS
BRITISH GRAND PRIX
 OFFICIAL RACE REPORT & STATISTICAL REVIEW
 
Circuit
Silverstone Circuit, Northamptonshire, United Kingdom
Race Date
Sunday, 7 July 2024
Round
Round 12 of 24 — 2024 FIA Formula One World Championship
Distance
52 laps × 5.891 km = 306.198 km
Weather
Partly cloudy, 21°C ambient, 38°C track surface
Attendance
480,000 (race weekend total — record for Silverstone)
Document Ref
F1-2024-GBR-RACE-REPORT-v1.2
This document is published by the FIA Formula One Administration and is intended for official championship and media
purposes. All lap time data sourced from official FIA transponder systems. Tyre compound information provided by Pirelli
Motorsport.


In [9]:
#Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

#create ChromaDB and store chunks
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "chroma_db"
)

print(f"Total vectors stored :{vector_store._collection.count()}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Total vectors stored :16


In [10]:
#Ask the question
query = "Who won the 2024 British Grand Prix?"

#Search ChromaDB for the most relevent chunks
results = vector_store.similarity_search(
    query = query,
    k=3
)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ----")
    print(doc.page_content)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



--- Chunk 1 ----
1. RACE NARRATIVE & MATCH REPORT
1.1 Pre-Race Grid & Starting Conditions
The 2024 British Grand Prix commenced under overcast skies at Silverstone, with ambient temperatures of
21°C and a track surface temperature of 38°C — considerably cooler than forecast and a significant factor in
tyre degradation profiles throughout the afternoon. Lando Norris (McLaren, Car #4) secured pole position in a
dramatic qualifying session on Saturday, posting a lap of 1:25.912 — his second career pole and the fastest
ever lap recorded at Silverstone in Formula One. Max Verstappen (Red Bull Racing, Car #1) qualified second,
0.228s adrift, with Carlos Sainz (Ferrari, Car #55) completing the front row after a late improvement on his final
flying lap.
Mercedes placed both cars in the top six: Lewis Hamilton qualified fourth (1:26.310) ahead of teammate George
Russell in fifth (1:26.489). Oscar Piastri (McLaren, Car #81) started seventh after a wheel-sensor anomaly

--- Chunk 2 ----
calendar